# 01 Bronze: raw ingestion

Statements of Reasons (SoRs) that TikTok and X submitted to the
[DSA Transparency Database](https://transparency.dsa.ec.europa.eu/) between 2025-01-01 and 2025-12-31.
One row = one statement of reasons (one moderation decision).

| Layer | Notebook | Content | Storage |
|---|---|---|---|
| **Bronze** | `01_bronze.ipynb` | Raw SoRs exactly as published, plus lineage columns | Parquet chunks from `dsa-tdb` (read-only), exposed as DuckDB views |
| **Silver** | `02_silver.ipynb` | Scoped, cleaned, typed SoRs | `E:/dsa-data/silver/` (parquet) |
| **Gold** | `03_gold.ipynb` | Analysis-ready tables | *tbd* |

Paths, constants and the DuckDB connection are shared through `config.py`.
All processing runs in **DuckDB** (out-of-core): TikTok alone has ~1 billion rows, far more than fits in memory.

## Setup

In [1]:
from config import DATA_ROOT, PLATFORMS, PERIOD, connect

import io
import zipfile
from datetime import date, timedelta

import pandas as pd

con = connect()
pd.set_option("display.max_rows", 100)

## 1. Source and conversion to parquet

The daily dumps were downloaded from the [data download page](https://transparency.dsa.ec.europa.eu/data-download)
and converted with the Commission's official tool [`dsa-tdb`](https://code.europa.eu/dsa/transparency-database/dsa-tdb),
run as its Docker stack (image `code.europa.eu:4567/dsa/transparency-database/dsa-tdb:latest`). The conversion was started
from the tool's web app (`prepare` task), with the data folder mounted into the container as `/data`.

For each day, the task:
1. downloads `sor-<platform>-<date>-full.zip` and verifies its SHA1 checksum,
2. unpacks the nested CSV zips and rewrites them as parquet files of at most 1M rows,
3. writes a `COMPLETE` marker once the day is done.

Resulting layout (data folder now at `E:/dsa-data`):
```
<platform>___full/
├── parameters.yaml                        # settings of the dsa-tdb run
├── sor-<platform>-<date>-full.zip(.sha1)  # original daily dumps (kept)
└── daily_dumps_chunked/sor-<platform>-<date>-full/
    ├── COMPLETE
    └── part-0000.parquet, part-0001.parquet, ...
```

`dsa-tdb` writes the settings of each run to `parameters.yaml`; this is the record of how the data was produced
(paths refer to the container, e.g. `/data/tiktok___full`):

In [2]:
for p in PLATFORMS:
    print(f"--- {p}/parameters.yaml ---")
    print((DATA_ROOT / f"{p}___full" / "parameters.yaml").read_text())

--- tiktok/parameters.yaml ---
check_sha1: true
chunk_format: parquet
chunk_size: 1000000
delete_original: false
do_chunking: true
dump_files_folder: /data/tiktok___full
force_sha1: false
from_date: 2025-01-01
loglevel: 20
n_processes: 1
override_chunked_subfolder: daily_dumps_chunked
platform: tiktok
platforms_to_exclude: null
raise_on_error: true
to_date: 2025-12-31
version: full

--- x/parameters.yaml ---
check_sha1: true
chunk_format: parquet
chunk_size: 1000000
delete_original: false
do_chunking: true
dump_files_folder: /data/x___full
force_sha1: false
from_date: 2025-01-01
loglevel: 20
n_processes: 1
override_chunked_subfolder: daily_dumps_chunked
platform: x
platforms_to_exclude: null
raise_on_error: true
to_date: 2025-12-31
version: full



## 2. Completeness check

For every day in 2025, check that the zip exists, that `dsa-tdb` finished it (`COMPLETE`), and whether it produced parquet files.

In [3]:
def day_status(platform: str) -> pd.DataFrame:
    base = DATA_ROOT / f"{platform}___full"
    rows = []
    d = PERIOD[0]
    while d <= PERIOD[1]:
        name = f"sor-{platform}-{d}-full"
        chunk_dir = base / "daily_dumps_chunked" / name
        rows.append({
            "platform": platform,
            "date": d,
            "zip": (base / f"{name}.zip").exists(),
            "complete": (chunk_dir / "COMPLETE").exists(),
            "parquet_files": len(list(chunk_dir.glob("*.parquet"))),
        })
        d += timedelta(days=1)
    return pd.DataFrame(rows)


days = pd.concat([day_status(p) for p in PLATFORMS], ignore_index=True)
days.groupby("platform").agg(
    days=("date", "size"),
    zips=("zip", "sum"),
    complete=("complete", "sum"),
    days_with_data=("parquet_files", lambda s: (s > 0).sum()),
    days_without_data=("parquet_files", lambda s: (s == 0).sum()),
    parquet_files=("parquet_files", "sum"),
)

,days,zips,complete,days_with_data,days_without_data,parquet_files
platform,,,,,,
tiktok,365,365,365,329,36,1197
x,365,365,365,291,74,291


Some days have no parquet output. The next cell looks inside the original zips of those days to confirm
they are empty **in the source** (the CSVs contain only a header) and not a failed conversion.

In [4]:
def data_lines_in_zip(zip_path: Path) -> int:
    """Count CSV rows (excluding headers) across the nested csv.zip files of a daily dump."""
    n = 0
    with zipfile.ZipFile(zip_path) as outer:
        for inner_name in outer.namelist():
            with zipfile.ZipFile(io.BytesIO(outer.read(inner_name))) as inner:
                for csv_name in inner.namelist():
                    n += max(inner.read(csv_name).count(b"\n") - 1, 0)
    return n


def to_ranges(ds: list[date]) -> list[str]:
    ranges, start = [], ds[0]
    for prev, cur in zip(ds, ds[1:] + [None]):
        if cur is None or cur - prev > timedelta(days=1):
            ranges.append(f"{start}" if start == prev else f"{start} to {prev}")
            start = cur
    return ranges


for p in PLATFORMS:
    empty = days.query("platform == @p and parquet_files == 0")["date"].tolist()
    rows_in_source = sum(data_lines_in_zip(DATA_ROOT / f"{p}___full" / f"sor-{p}-{d}-full.zip") for d in empty)
    print(f"{p}: {len(empty)} days without data, rows in their source zips: {rows_in_source}")
    for r in to_ranges(empty):
        print("   ", r)

tiktok: 36 days without data, rows in their source zips: 0
    2025-09-05 to 2025-10-09
    2025-12-03
x: 74 days without data, rows in their source zips: 0
    2025-03-25 to 2025-04-15
    2025-07-02 to 2025-07-14
    2025-07-16 to 2025-08-09
    2025-08-11 to 2025-08-16
    2025-08-18 to 2025-08-25


## 3. Register bronze views

The parquet chunks already are the raw data, so bronze does **not** copy ~1B rows. It registers one DuckDB view per
platform over the chunk files (data stays read-only on disk). Two lineage columns are added:
- `dump_date`: date of the daily dump the row came from (parsed from the folder name)
- `source_file`: parquet file the row was read from

DuckDB only reads the files, columns and row groups a query needs, so filtering on `dump_date` or selecting a few columns stays fast.

In [5]:
con.execute("CREATE SCHEMA IF NOT EXISTS bronze")

for p in PLATFORMS:
    glob = (DATA_ROOT / f"{p}___full" / "daily_dumps_chunked" / "*" / "*.parquet").as_posix()
    con.execute(f"""
        CREATE OR REPLACE VIEW bronze.{p} AS
        SELECT
            * EXCLUDE (filename),
            regexp_extract(filename, '(\\d{{4}}-\\d{{2}}-\\d{{2}})', 1)::DATE AS dump_date,
            filename AS source_file
        FROM read_parquet('{glob}', filename = true)
    """)

con.sql("SELECT table_schema, table_name FROM information_schema.tables WHERE table_schema = 'bronze'").df()

,table_schema,table_name
0,bronze,tiktok
1,bronze,x


## 4. Raw data overview

Rows and columns ingested per platform. Column counts exclude the two lineage columns.

In [6]:
LINEAGE_COLS = ["dump_date", "source_file"]

overview = []
for p in PLATFORMS:
    n_rows = con.sql(f"SELECT count(*) FROM bronze.{p}").fetchone()[0]
    cols = [c for c in con.sql(f"SELECT * FROM bronze.{p} LIMIT 0").columns if c not in LINEAGE_COLS]
    overview.append({"platform": p, "rows": f"{n_rows:,}", "columns": len(cols)})

pd.DataFrame(overview)

,platform,rows,columns
0,tiktok,"1,002,729,268",37
1,x,"670,093",37


Raw schema (identical for both platforms):

In [7]:
schemas = {p: con.sql(f"DESCRIBE bronze.{p}").df()[["column_name", "column_type"]] for p in PLATFORMS}
assert schemas["tiktok"].equals(schemas["x"]), "schemas differ between platforms"

raw_schema = schemas["tiktok"].query("column_name not in @LINEAGE_COLS").reset_index(drop=True)
raw_schema

,column_name,column_type
0,uuid,VARCHAR
1,decision_visibility,VARCHAR
2,decision_visibility_other,VARCHAR
3,end_date_visibility_restriction,TIMESTAMP
4,decision_monetary,VARCHAR
5,decision_monetary_other,VARCHAR
6,end_date_monetary_restriction,TIMESTAMP
7,decision_provision,VARCHAR
8,end_date_service_restriction,TIMESTAMP
9,decision_account,VARCHAR


## 5. Profiling

Checks on the raw data that inform the Silver rules. Nothing is changed here.

#### Empty columns

Columns that contain no values at all (100% `NULL` over all rows) per platform. These carry no information
and are candidates for removal in Silver. Requires one full scan of each platform.

In [9]:
raw_cols = raw_schema["column_name"].tolist()

for p in PLATFORMS:
    non_null = con.sql(
        f"SELECT {', '.join(f'count({c}) AS {c}' for c in raw_cols)} FROM bronze.{p}"
    ).df().iloc[0]
    all_null = non_null[non_null == 0].index.tolist()
    print(f"{p}: {len(all_null)} of {len(raw_cols)} columns are 100% NULL")
    for c in all_null:
        print("   ", c)

tiktok: 9 of 37 columns are 100% NULL
    decision_monetary_other
    account_type
    decision_ground_reference_url
    incompatible_content_illegal
    category_addition
    category_specification
    category_specification_other
    content_language
    source_identity
x: 14 of 37 columns are 100% NULL
    end_date_visibility_restriction
    decision_monetary
    decision_monetary_other
    end_date_monetary_restriction
    end_date_service_restriction
    end_date_account_restriction
    account_type
    decision_ground_reference_url
    incompatible_content_ground
    incompatible_content_explanation
    incompatible_content_illegal
    content_type_other
    content_language
    source_identity


Filled in X, but 100% NULL in TikTok (3 columns):

category_addition\
category_specification\
category_specification_other\
#======================================================

Filled in TikTok, but 100% NULL in X (8 columns):

end_date_visibility_restriction\
decision_monetary\
end_date_monetary_restriction\
end_date_service_restriction\
end_date_account_restriction\
incompatible_content_ground\
incompatible_content_explanation\
content_type_other\
#======================================================

100% NULL in Both TikTok and X (6 columns):

decision_monetary_other\
account_type\
decision_ground_reference_url\
incompatible_content_illegal\
content_language\
source_identity

#### Scope columns

Values of the columns the thesis scope is based on, checked before the scope rules are defined in Silver.

**`territorial_scope`**: stored as a JSON list of country codes per statement (e.g. `["DE"]`). The distinct lists are
counted first (few distinct values), then split into single countries.

In [2]:
scope_lists = con.sql(" UNION ALL ".join(
    f"SELECT '{p}' AS platform, territorial_scope, count(*) AS n FROM bronze.{p} GROUP BY ALL" for p in PLATFORMS
)).df()

print("Distinct lists and statements without territorial_scope:")
display(con.sql("""
    SELECT platform,
           count(territorial_scope) AS distinct_lists,
           coalesce(sum(n) FILTER (WHERE territorial_scope IS NULL OR territorial_scope = '[]'), 0) AS rows_without_scope
    FROM scope_lists GROUP BY platform ORDER BY platform
""").df())

scope_countries = con.sql("""
    SELECT platform, n, unnest(from_json(territorial_scope, '["VARCHAR"]')) AS country
    FROM scope_lists
""")

print("Rows per country (a statement is counted for every country in its list):")
display(con.sql("""
    PIVOT (SELECT country, platform, sum(n) AS n FROM scope_countries GROUP BY ALL)
    ON platform USING sum(n) ORDER BY country
""").df().fillna(0).astype({p: "int64" for p in PLATFORMS}))

print("Number of countries per statement:")
display(con.sql("""
    PIVOT (SELECT json_array_length(territorial_scope) AS countries_in_list, platform, n FROM scope_lists)
    ON platform USING sum(n) ORDER BY countries_in_list
""").df().fillna(0).astype({p: "int64" for p in PLATFORMS}))

Distinct lists and statements without territorial_scope:


,platform,distinct_lists,rows_without_scope
0,tiktok,277046,4646.0
1,x,27,0.0


Rows per country (a statement is counted for every country in its list):


,country,tiktok,x
0,AT,995291158,7833
1,BE,995380455,12211
2,BG,994160590,6004
3,CY,993911800,2290
4,CZ,994689328,11594
5,DE,999079277,183324
6,DK,995272136,6022
7,EE,994227778,3161
8,ES,995820122,66757
9,FI,995098010,6133


Number of countries per statement:


,countries_in_list,tiktok,x
0,0,4646,0
1,1,6274629,670093
2,2,302306,0
3,3,224387,0
4,4,140677,0
5,5,137870,0
6,6,120767,0
7,7,135983,0
8,8,103930,0
9,9,97742,0


+ TikTok almost always lists the whole EEA. 983M statements list 29 countries (EU-27 plus Liechtenstein and Norway), and 9.7M list 30 (plus Iceland). Only 6.3M statements name a single country, and 4,646 have an empty list. TikTok uses 277,046 different country lists.
+ X always names exactly one country. Germany is the largest (183,324), then France (136,338) and Spain (66,757).
* Consequence for your scope: a filter like "territorial_scope contains DE" keeps about 999M TikTok rows but only 183K X rows. The two platforms don't mean the same thing by this field, so "targets Germany" and "applies in Germany, among other countries" have to be decided separately.

**`platform_name`**

In [3]:
con.sql(" UNION ALL ".join(
    f"SELECT '{p}' AS platform, platform_name, count(*) AS n FROM bronze.{p} GROUP BY ALL" for p in PLATFORMS
) + " ORDER BY platform").df()

,platform,platform_name,n
0,tiktok,TikTok,1002729268
1,x,X,668624
2,x,X (formerly Twitter),1469


**Date columns**: earliest and latest value per platform.

In [4]:
date_ranges = []
for p in PLATFORMS:
    date_cols = con.sql(f"DESCRIBE bronze.{p}").df().query("column_type == 'TIMESTAMP'")["column_name"].tolist()
    row = con.sql(
        f"SELECT {', '.join(f'min({c}), max({c})' for c in date_cols)} FROM bronze.{p}"
    ).fetchone()
    for i, c in enumerate(date_cols):
        date_ranges.append({"column": c, "platform": p, "min": row[2 * i], "max": row[2 * i + 1]})

pd.DataFrame(date_ranges).sort_values(["column", "platform"]).reset_index(drop=True)

,column,platform,min,max
0,application_date,tiktok,2020-01-01 00:00:00,2025-12-12 00:00:00
1,application_date,x,2024-12-31 00:00:00,2025-12-31 00:00:00
2,content_date,tiktok,2000-01-01 00:00:00,2025-12-18 00:00:00
3,content_date,x,2024-12-31 00:00:00,2025-12-31 00:00:00
4,created_at,tiktok,2025-01-01 00:00:06,2025-12-31 23:59:59
5,created_at,x,2025-01-01 00:47:03,2025-12-31 23:56:16
6,end_date_account_restriction,tiktok,2023-02-27 00:00:00,2026-06-25 00:00:00
7,end_date_account_restriction,x,NaT,NaT
8,end_date_monetary_restriction,tiktok,2025-01-23 00:00:00,2028-06-24 00:00:00
9,end_date_monetary_restriction,x,NaT,NaT
